# 🐾 PetMind: Sistema Agéntico de Recomendación Nutricional y Gestión de Dietas
### Asignatura: Ingeniería de Soluciones con IA (ISY0101) - Evaluación Parcial N°2
**Caso de Aplicación:** NutriPet Catapilco  
**Integrantes:** Gustavo Giha — Matias Morales  
**Frameworks Principales:** `LangGraph`, `LangChain`, `ChatGoogleGenerativeAI (Gemini)`, `FAISS`, `Pydantic v2`  
---
### Objetivos e Indicadores de Logro:
- **IL2.1 (IE1, IE2):** Agente funcional con herramientas tipadas de Consulta (catálogo RAG), Escritura (registro de órdenes) y Razonamiento (validación clínica).
- **IL2.2 (IE3, IE4):** Memoria de corto plazo (STM - ventana episódica) y memoria de largo plazo (LTM - perfiles y guías clínicas vectorizadas).
- **IL2.3 (IE5, IE7):** Planificación adaptativa y enrutamiento dinámico ante condiciones cambiantes (alertas biológicas vs compra vs asesoría).
- **IL2.4 (IE6, IE8):** Orquestación mediante StateGraph determinista, trazable y auditable.

## 1. Instalación de Dependencias y Configuración del Entorno
Instalamos las librerías oficiales utilizadas a lo largo de las sesiones 8 a 11.

In [ ]:
# Instalación de librerías requeridas
!pip install -q --upgrade langchain langchain-core langchain-community langchain-google-genai langgraph faiss-cpu pydantic tabulate


## 2. Autenticación con Google Gemini API
Configuración segura de la API Key mediante `userdata` de Google Colab (o variable de entorno en entornos locales).

In [ ]:
import os
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Clave GOOGLE_API_KEY cargada exitosamente desde Secrets de Colab.")
except Exception:
    if "GOOGLE_API_KEY" in os.environ:
        print("✅ Clave GOOGLE_API_KEY detectada en el entorno local.")
    else:
        import getpass
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key: ")


## 3. Inicialización del Modelo Fundacional (Gemini Flash)
Instanciamos `ChatGoogleGenerativeAI` con temperatura 0.2 para mitigar alucinaciones y favorecer el determinismo técnico.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Modelo generativo de razonamiento
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

# Modelo de embeddings semánticos para memoria LTM
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
print("✅ LLM y Embeddings instanciados correctamente.")


## 4. Fuentes de Datos y Memoria de Largo Plazo (LTM Semántica - IE4)
Indexamos en memoria vectorial FAISS:
1. **Catálogo de Productos de NutriPet Catapilco** (alimentos caninos y felinos con composición garantizada).
2. **Guías Clínicas Veterinarias (VCA Animal Hospitals)** (taurina, relación calcio-fósforo, prevención renal).

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Base de datos simulada del catálogo local (NutriPet Catapilco)
CATALOGO_PRODUCTOS = [
    Document(
        page_content="SKU: NP-DOG-001 | NutriPuppy Giant Growth Formula | Especie: perro | Etapa: cachorro | Raza: gigante | Formato: seco | Precio: $46.990 CLP | Stock: 18 | Proteína: 30% | Calcio: 1.1% | Fósforo: 0.9% | Descripción: Balance estricto Ca:P (1.2:1) con condroitina para prevenir displasia y osteocondritis en cachorros de rápido crecimiento.",
        metadata={"sku": "NP-DOG-001", "especie": "perro", "etapa": "cachorro", "precio": 46990, "stock": 18}
    ),
    Document(
        page_content="SKU: NP-DOG-002 | NutriDog Adult Medium Maintenance | Especie: perro | Etapa: adulto | Raza: mediano | Formato: seco | Precio: $38.990 CLP | Stock: 25 | Proteína: 25% | Calcio: 1.2% | Fósforo: 1.0% | Descripción: Mantenimiento diario completo con pulpa de remolacha y prebióticos MOS para digestión sana en perros adultos de 10 a 25 kg.",
        metadata={"sku": "NP-DOG-002", "especie": "perro", "etapa": "adulto", "precio": 38990, "stock": 25}
    ),
    Document(
        page_content="SKU: NP-DOG-003 | NutriDog Senior Mobility & Renal Support | Especie: perro | Etapa: senior | Raza: todos | Formato: seco | Precio: $42.990 CLP | Stock: 12 | Proteína: 22% | Calcio: 0.8% | Fósforo: 0.5% | Descripción: Restricción controlada de fósforo para protección renal en perros mayores a 7 años, enriquecido con EPA/DHA y glucosamina para movilidad.",
        metadata={"sku": "NP-DOG-003", "especie": "perro", "etapa": "senior", "precio": 42990, "stock": 12}
    ),
    Document(
        page_content="SKU: NP-CAT-001 | NutriKitten Vital Defense & DHA | Especie: gato | Etapa: cachorro | Raza: todos | Formato: seco | Precio: $34.990 CLP | Stock: 15 | Proteína: 36% | Calcio: 1.2% | Fósforo: 1.0% | Taurina: 2500 mg/kg | Descripción: Alto en proteína animal y reforzado con 2500 mg/kg de taurina para desarrollo de retina y sistema cardíaco en gatitos.",
        metadata={"sku": "NP-CAT-001", "especie": "gato", "etapa": "cachorro", "precio": 34990, "stock": 15}
    ),
    Document(
        page_content="SKU: NP-CAT-002 | NutriCat Adult Sterilized Weight Control | Especie: gato | Etapa: adulto | Raza: todos | Formato: seco | Precio: $39.990 CLP | Stock: 20 | Proteína: 34% | Grasa: 10% | Taurina: 2400 mg/kg | Descripción: Fórmula con control de pH urinario y L-carnitina para prevenir litiasis urinaria y sobrepeso en felinos esterilizados.",
        metadata={"sku": "NP-CAT-002", "especie": "gato", "etapa": "adulto", "precio": 39990, "stock": 20}
    ),
    Document(
        page_content="SKU: NP-CAT-003 | NutriCat Wet Gourmet Pouch Salmón & Taurina | Especie: gato | Etapa: adulto | Raza: todos | Formato: humedo | Precio: $1.490 CLP | Stock: 80 | Proteína: 11% | Humedad: 82% | Taurina: 3000 mg/kg | Descripción: Alimento húmedo premium con alto aporte hídrico para prevenir litiasis y enfermedad renal felina.",
        metadata={"sku": "NP-CAT-003", "especie": "gato", "etapa": "adulto", "precio": 1490, "stock": 80}
    )
]

# Guías clínicas veterinarias VCA Hospitals
GUIAS_VETERINARIAS = [
    Document(
        page_content="PROTOCOLO VCA-TAURINA: Los gatos son carnívoros estrictos y no sintetizan taurina. La carencia de taurina produce cardiomiopatía dilatada irreversible y degeneración central de retina (ceguera). JAMÁS se debe prescribir o suministrar alimento de perro a gatos, ya que carece del requerimiento taurínico mínimo.",
        metadata={"tipo": "alerta_clinica", "tema": "taurina_felina"}
    ),
    Document(
        page_content="PROTOCOLO VCA-CALCIO-FOSFORO: En cachorros de razas grandes y gigantes, el exceso de calcio (>1.4%) provoca osteocondritis disecante (OCD) y displasia esquelética. El balance Ca:P debe mantenerse estricto en torno a 1.1:1 a 1.2:1 con densidad energética moderada.",
        metadata={"tipo": "alerta_clinica", "tema": "crecimiento_gigante"}
    ),
    Document(
        page_content="PROTOCOLO VCA-RENAL-SENIOR: En caninos senior mayores a 7 años, una reducción del fósforo dietario disminuye la progresión de nefropatías crónicas, complementado con ácidos grasos Omega-3 (EPA/DHA) para proteger la microvasculatura renal.",
        metadata={"tipo": "alerta_clinica", "tema": "salud_renal_senior"}
    )
]

# Creación de vectorstores locales FAISS
vectorstore_catalogo = FAISS.from_documents(CATALOGO_PRODUCTOS, embeddings)
vectorstore_guias = FAISS.from_documents(GUIAS_VETERINARIAS, embeddings)
print("✅ LTM Vectorial indexada exitosamente con Catálogo y Guías Clínicas.")


## 5. Implementación de Herramientas Tipadas (Tools - IE1, IE2)
Siguiendo la metodología de la Sesión 10, implementamos subclases de `BaseTool` con esquemas estrictos en `Pydantic v2`:
- **Herramienta 1 (Consulta):** `ConsultarCatalogoTool` (recuperación RAG semántica del catálogo).
- **Herramienta 2 (Razonamiento / Validación Médica):** `ValidarSeguridadNutricionalTool` (evaluación de riesgos fisiológicos y guardrails).
- **Herramienta 3 (Escritura / Transaccional):** `RegistrarOrdenCompraTool` (reserva y persistencia auditable de pedidos).

In [ ]:
from typing import Type, Optional
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool

# Registro en memoria mutable para simular inventario y base de pedidos
ORDENES_REGISTRADAS = []

# --- 1. HERRAMIENTA DE CONSULTA ---
class ConsultaCatalogoInput(BaseModel):
    query: str = Field(description="Término de búsqueda clínica o nutricional, ej: cachorro gigante, gato esterilizado")
    k: int = Field(default=2, description="Número de productos candidatos a recuperar")

class ConsultarCatalogoTool(BaseTool):
    name: str = "consultar_catalogo_nutripet"
    description: str = "Consulta el inventario y especificaciones técnicas de alimentos disponibles en NutriPet Catapilco."
    args_schema: Type[BaseModel] = ConsultaCatalogoInput

    def _run(self, query: str, k: int = 2) -> str:
        resultados = vectorstore_catalogo.similarity_search(query, k=k)
        if not resultados:
            return "No se encontraron productos coincidentes en el catálogo de NutriPet."
        return "\n---\n".join([doc.page_content for doc in resultados])

# --- 2. HERRAMIENTA DE RAZONAMIENTO ---
class ValidarSeguridadInput(BaseModel):
    especie: str = Field(description="Especie destino: perro o gato")
    etapa: str = Field(description="Etapa fisiológica: cachorro, adulto, senior")
    alimento_destinado: str = Field(description="Tipo de alimento que el cliente pretende suministrar")

class ValidarSeguridadNutricionalTool(BaseTool):
    name: str = "validar_seguridad_nutricional"
    description: str = "Razona y valida si la dieta solicitada presenta riesgos fisiológicos graves (taurina, displasia, etc)."
    args_schema: Type[BaseModel] = ValidarSeguridadInput

    def _run(self, especie: str, etapa: str, alimento_destinado: str) -> str:
        esp = especie.lower().strip()
        alim = alimento_destinado.lower().strip()
        if "gato" in esp and "perro" in alim:
            guia = vectorstore_guias.similarity_search("taurina deficiencia gato", k=1)[0]
            return f"🚨 ALERTA BIOLÓGICA CRÍTICA: {guia.page_content} -> INCOMPATIBILIDAD VETERINARIA. RECHAZAR SOLICITUD."
        if "perro" in esp and "cachorro" in etapa.lower() and ("gigante" in alim or "grande" in alim):
            guia = vectorstore_guias.similarity_search("calcio fosforo cachorro gigante", k=1)[0]
            return f"⚠️ REQUISITO CLÍNICO: {guia.page_content} -> VERIFICAR RELACIÓN Ca:P RESTRINGIDA (NP-DOG-001)."
        return "✅ DIETA SEGURA: No se detectan contraindicaciones biológicas graves."

# --- 3. HERRAMIENTA DE ESCRITURA ---
class RegistroOrdenInput(BaseModel):
    cliente_nombre: str = Field(description="Nombre del cliente tutor de la mascota")
    sku: str = Field(description="Código SKU del producto a ordenar")
    cantidad: int = Field(default=1, description="Cantidad de unidades a comprar")
    mascota_nombre: Optional[str] = Field(default="Mascota", description="Nombre del animal para ficha técnica")

class RegistrarOrdenCompraTool(BaseTool):
    name: str = "registrar_orden_compra"
    description: str = "Herramienta de escritura para persistir una orden de compra formal en NutriPet Catapilco."
    args_schema: Type[BaseModel] = RegistroOrdenInput

    def _run(self, cliente_nombre: str, sku: str, cantidad: int = 1, mascota_nombre: str = "Mascota") -> str:
        import uuid
        orden_id = f"ORD-{uuid.uuid4().hex[:6].upper()}"
        registro = {
            "orden_id": orden_id,
            "cliente": cliente_nombre,
            "mascota": mascota_nombre,
            "sku": sku,
            "cantidad": cantidad,
            "estado": "CONFIRMADA_AUDITADA"
        }
        ORDENES_REGISTRADAS.append(registro)
        return f"✅ ORDEN REGISTRADA EXITOSAMENTE: [ID: {orden_id}] - {cantidad}x {sku} para '{mascota_nombre}' (Tutor: {cliente_nombre})."

tool_consulta = ConsultarCatalogoTool()
tool_razonamiento = ValidarSeguridadNutricionalTool()
tool_escritura = RegistrarOrdenCompraTool()
herramientas = [tool_consulta, tool_razonamiento, tool_escritura]
print(f"✅ Herramientas configuradas: {[t.name for t in herramientas]}")


## 6. Arquitectura del Agente con LangGraph: Estado, Memoria y Planificación (IE3, IE5)
Definimos el grafo de estado compartido `EstadoPetMind` que incorpora memoria de corto y largo plazo con enrutamiento dinámico.

In [ ]:
from typing import List, Dict, Any, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

class EstadoPetMind(BaseModel):
    messages: Annotated[List[BaseMessage], add_messages] = Field(default_factory=list)
    perfil_mascota: Dict[str, Any] = Field(default_factory=dict)
    alerta_critica: Optional[str] = Field(default=None)
    intencion_actual: str = Field(default="asesoria")
    recomendacion_final: Optional[str] = Field(default=None)

print("✅ Esquema EstadoPetMind compilado.")


### 6.1 Nodos del Grafo de Decisión (Planificación Adaptativa)

In [ ]:
def nodo_planificador_intencion(estado: EstadoPetMind) -> Dict[str, Any]:
    ultimo_mensaje = estado.messages[-1].content.lower()
    if any(k in ultimo_mensaje for k in ["comprar", "ordenar", "llevar", "pedido", "confirmar"]):
        intencion = "compra"
    elif any(k in ultimo_mensaje for k in ["darle a mi gato comida de perro", "dar de perro a gato", "sobras", "incompatible"]):
        intencion = "evaluar_seguridad"
    else:
        intencion = "asesoria"
    return {"intencion_actual": intencion}

def nodo_guardrail_clinico(estado: EstadoPetMind) -> Dict[str, Any]:
    ultimo_texto = estado.messages[-1].content
    resultado = tool_razonamiento.run({
        "especie": "gato" if "gato" in ultimo_texto.lower() else "perro",
        "etapa": "cachorro" if "cachorro" in ultimo_texto.lower() else "adulto",
        "alimento_destinado": ultimo_texto
    })
    if "ALERTA BIOLÓGICA" in resultado:
        msg = AIMessage(content=f"{resultado}\n\n🛑 Por razones de bienestar animal en NutriPet Catapilco, NO es posible autorizar esta recomendación. Te sugerimos revisar las fórmulas felinas específicas con taurina activa.")
        return {"alerta_critica": resultado, "messages": [msg], "recomendacion_final": msg.content}
    return {"alerta_critica": None}

def nodo_asesor_nutricional(estado: EstadoPetMind) -> Dict[str, Any]:
    consulta = estado.messages[-1].content
    contexto_catalogo = tool_consulta.run({"query": consulta, "k": 2})
    prompt_sistema = (
        "Eres PetMind, el especialista en nutrición clínica de NutriPet Catapilco.\n"
        "Responde de forma clara, técnica y comercial. Cita obligatoriamente SKU, Precio CLP y Justificación Nutricional.\n"
        f"CATÁLOGO DISPONIBLE:\n{contexto_catalogo}\n"
    )
    respuesta_llm = llm.invoke([
        SystemMessage(content=prompt_sistema),
        HumanMessage(content=consulta)
    ])
    return {"messages": [respuesta_llm], "recomendacion_final": respuesta_llm.content}

def nodo_gestor_ordenes(estado: EstadoPetMind) -> Dict[str, Any]:
    consulta = estado.messages[-1].content
    sku_detectado = "NP-DOG-001" if "dog" in consulta.lower() or "perro" in consulta.lower() else "NP-CAT-001"
    resultado_escritura = tool_escritura.run({
        "cliente_nombre": "Gustavo Giha",
        "sku": sku_detectado,
        "cantidad": 1,
        "mascota_nombre": "Max"
    })
    msg = AIMessage(content=f"{resultado_escritura}\nTu orden ha sido registrada en el sistema de mesón de NutriPet Catapilco. Puedes pasar a retirar en tienda.")
    return {"messages": [msg], "recomendacion_final": msg.content}
print("✅ Nodos del grafo definidos.")


### 6.2 Construcción y Compilación del StateGraph

In [ ]:
def ruteador_planificacion(estado: EstadoPetMind) -> str:
    if estado.intencion_actual == "evaluar_seguridad":
        return "guardrail"
    elif estado.intencion_actual == "compra":
        return "ordenes"
    return "asesoria"

builder = StateGraph(EstadoPetMind)
builder.add_node("planificador", nodo_planificador_intencion)
builder.add_node("guardrail", nodo_guardrail_clinico)
builder.add_node("asesor", nodo_asesor_nutricional)
builder.add_node("ordenes", nodo_gestor_ordenes)

builder.add_edge(START, "planificador")
builder.add_conditional_edges(
    "planificador",
    ruteador_planificacion,
    {
        "guardrail": "guardrail",
        "asesoria": "asesor",
        "ordenes": "ordenes"
    }
)
builder.add_edge("guardrail", END)
builder.add_edge("asesor", END)
builder.add_edge("ordenes", END)

agente_petmind = builder.compile()
print("🚀 Agente PetMind compilado y listo para ejecución.")


## 7. Batería de Pruebas y Casos de Demostración en Vivo (IE7)
Validamos los 3 escenarios organizacionales:
1. **Escenario 1 (Asesoría Clínica RAG):** Cachorro gigante y balance Ca:P.
2. **Escenario 2 (Guardrail de Riesgo Biológico):** Intento de alimentar gatos con comida canina.
3. **Escenario 3 (Transaccional / Escritura):** Registro de orden de compra formal.

In [ ]:
def probar_agente(pregunta: str):
    print("\n" + "="*70)
    print(f"👤 TUTOR: {pregunta}")
    print("="*70)
    estado_inicial = {"messages": [HumanMessage(content=pregunta)]}
    salida = agente_petmind.invoke(estado_inicial)
    print(f"🐾 PETMIND:\n{salida['recomendacion_final']}")

# Prueba 1: Cachorro Gigante
probar_agente("Tengo un cachorro Gran Danés de 3 meses. ¿Qué alimento le doy para cuidar sus articulaciones?")

# Prueba 2: Incompatibilidad Crítica (Taurina)
probar_agente("¿Puedo darle a mi gato la comida de perro para ahorrar presupuesto?")

# Prueba 3: Cierre de Orden
probar_agente("Quiero confirmar y comprar una bolsa de alimento para mi perro")


## 8. Evidencia de Auditoría Transaccional
Mostramos las órdenes persistidas en el sistema de NutriPet Catapilco.

In [ ]:
import pandas as pd
df_ordenes = pd.DataFrame(ORDENES_REGISTRADAS)
print("📋 REGISTRO AUDITABLE DE ÓRDENES EN NUTRIPET CATAPILCO:")
display(df_ordenes) if not df_ordenes.empty else print("No hay órdenes registradas.")
